# Feature Engineering for Steam Price Modeling

This notebook turns the raw Steam games dataset into a clean modeling table for market-aligned price prediction. It builds an estimated list-price target, separates pre-release and post-release feature groups, splits multi-label metadata, and saves a reusable CSV for later modeling notebooks.

In [ ]:
import ast
import csv
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "games.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"


## 1. Load and clean raw data

The raw CSV has a malformed header where `Discount` and `DLC count` are merged. The loader repairs that first, then applies basic type conversions and normalizes blank strings.

In [ ]:
RAW_DATA_PATH = DATA_DIR / "games.csv"
OUTPUT_PATH = DATA_DIR / "games_price_model.csv"

NUMERIC_COLS = [
    "Peak CCU",
    "Required age",
    "Price",
    "Discount",
    "DLC count",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Score rank",
    "Achievements",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
]

BOOL_COLS = ["Windows", "Mac", "Linux"]
TEXT_COLS = [
    "Name",
    "Estimated owners",
    "About the game",
    "Supported languages",
    "Full audio languages",
    "Reviews",
    "Website",
    "Support url",
    "Support email",
    "Metacritic url",
    "Developers",
    "Publishers",
    "Categories",
    "Genres",
    "Tags",
    "Screenshots",
    "Movies",
    "Notes",
]


def load_games_csv(path):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        raw_header = next(reader)
        header = raw_header[:7] + ["Discount", "DLC count"] + raw_header[8:]

        for raw_row in reader:
            if len(raw_row) != len(header):
                raise ValueError(f"Unexpected row length {len(raw_row)}")
            rows.append(raw_row)

    df = pd.DataFrame(rows, columns=header)

    for col in TEXT_COLS:
        if col in df.columns:
            df[col] = df[col].replace("", np.nan)

    df["AppID"] = pd.to_numeric(df["AppID"], errors="coerce").astype("Int64")
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in BOOL_COLS:
        df[col] = df[col].map({"True": True, "False": False})

    df["Release date parsed"] = pd.to_datetime(df["Release date"], format="%b %d, %Y", errors="coerce")
    df["Release year"] = df["Release date parsed"].dt.year

    return df


df_raw = load_games_csv(RAW_DATA_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 2. Build price target

`Price` is the current Steam price, which may already include an active discount. For market-aligned pricing, the main target should be estimated list price.

In [ ]:
df = df_raw.copy()

df["current_price"] = df["Price"]
df["has_active_discount"] = df["Discount"].gt(0)

conditions = [
    df["Price"].gt(0) & df["Discount"].eq(0),
    df["Price"].gt(0) & df["Discount"].gt(0) & df["Discount"].lt(100),
]
values = [
    df["Price"],
    df["Price"] / (1 - df["Discount"] / 100),
]

df["estimated_list_price"] = np.select(conditions, values, default=np.nan)
df["estimated_list_price"] = pd.Series(df["estimated_list_price"], index=df.index).round(2)

df["price_target_source"] = np.select(
    [
        df["Price"].gt(0) & df["Discount"].eq(0),
        df["Price"].gt(0) & df["Discount"].gt(0) & df["Discount"].lt(100),
        df["Price"].eq(0),
        df["Discount"].eq(100),
    ],
    ["observed", "reconstructed", "free", "invalid_discount_100"],
    default="invalid",
)

df["valid_price_target"] = df["estimated_list_price"].between(0.49, 100, inclusive="both")
df["log_list_price"] = np.where(df["valid_price_target"], np.log1p(df["estimated_list_price"]), np.nan)

target_summary = (
    df.groupby("price_target_source")
    .agg(
        games=("AppID", "size"),
        valid_targets=("valid_price_target", "sum"),
        median_current_price=("current_price", "median"),
        median_estimated_list_price=("estimated_list_price", "median"),
    )
    .sort_values("games", ascending=False)
)
target_summary

In [ ]:
print("Valid paid price targets:", df["valid_price_target"].sum())
print("Invalid Discount == 100 rows:", df["price_target_source"].eq("invalid_discount_100").sum())
print("Estimated list price > 100 rows:", df["estimated_list_price"].gt(100).sum())

df.loc[df["estimated_list_price"].gt(100), [
    "AppID", "Name", "current_price", "Discount", "estimated_list_price", "Genres", "Recommendations"
]].sort_values("estimated_list_price", ascending=False).head(20)

## 3. Engineer structured features

These are compact numeric and boolean fields that can be used directly or after scaling. Skewed counts are log-transformed and clipped versions are kept for robust baselines.

In [ ]:
REFERENCE_YEAR = 2026

df["platform_count"] = df[BOOL_COLS].sum(axis=1)
df["release_age_years"] = REFERENCE_YEAR - df["Release year"]
df.loc[df["release_age_years"].lt(0), "release_age_years"] = np.nan

df["review_count"] = df["Positive"].fillna(0) + df["Negative"].fillna(0)
df["positive_share"] = np.where(df["review_count"].gt(0), df["Positive"] / df["review_count"], np.nan)

df["dlc_count_clipped"] = df["DLC count"].clip(lower=0, upper=50)
df["achievements_clipped"] = df["Achievements"].clip(lower=0, upper=500)
df["required_age_clipped"] = df["Required age"].clip(lower=0, upper=21)

df["log1p_dlc_count"] = np.log1p(df["DLC count"].clip(lower=0))
df["log1p_achievements"] = np.log1p(df["Achievements"].clip(lower=0))
df["log1p_review_count"] = np.log1p(df["review_count"])
df["log1p_recommendations"] = np.log1p(df["Recommendations"].fillna(0))
df["log1p_peak_ccu"] = np.log1p(df["Peak CCU"].fillna(0))
df["log1p_avg_playtime_forever"] = np.log1p(df["Average playtime forever"].fillna(0))
df["log1p_median_playtime_forever"] = np.log1p(df["Median playtime forever"].fillna(0))

structured_cols = [
    "current_price", "estimated_list_price", "log_list_price",
    "has_active_discount", "valid_price_target", "platform_count",
    "Release year", "release_age_years", "required_age_clipped",
    "dlc_count_clipped", "achievements_clipped", "log1p_dlc_count",
    "log1p_achievements", "review_count", "positive_share",
    "log1p_review_count", "log1p_recommendations", "log1p_peak_ccu",
    "log1p_avg_playtime_forever", "log1p_median_playtime_forever",
]

df[structured_cols].describe(include="all").T

## 4. Split multi-label fields

`Genres`, `Tags`, and `Categories` are comma-separated multi-label fields. Splitting them into lists preserves shared labels across different combinations and prepares them for multi-hot encoding in the modeling notebook.

In [ ]:
def split_items(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


df["genre_list"] = df["Genres"].apply(split_items)
df["tag_list"] = df["Tags"].apply(split_items)
df["category_list"] = df["Categories"].apply(split_items)

df["genre_count"] = df["genre_list"].str.len()
df["tag_count"] = df["tag_list"].str.len()
df["category_count"] = df["category_list"].str.len()

df[["Name", "Genres", "genre_list", "Tags", "tag_list", "Categories", "category_list"]].head()

In [ ]:
def label_counts(series):
    return series.explode().loc[lambda s: s.notna() & s.ne("")].value_counts()


genre_counts = label_counts(df.loc[df["valid_price_target"], "genre_list"])
tag_counts = label_counts(df.loc[df["valid_price_target"], "tag_list"])
category_counts = label_counts(df.loc[df["valid_price_target"], "category_list"])

frequency_summary = pd.DataFrame({
    "field": ["Genres", "Tags", "Categories"],
    "unique_labels": [len(genre_counts), len(tag_counts), len(category_counts)],
    "labels_count_ge_50": [(genre_counts >= 50).sum(), (tag_counts >= 50).sum(), (category_counts >= 50).sum()],
    "labels_count_ge_100": [(genre_counts >= 100).sum(), (tag_counts >= 100).sum(), (category_counts >= 100).sum()],
    "labels_count_ge_300": [(genre_counts >= 300).sum(), (tag_counts >= 300).sum(), (category_counts >= 300).sum()],
})
frequency_summary

In [ ]:
RETAINED_TAG_MIN_COUNT = 100
RETAINED_CATEGORY_MIN_COUNT = 100

retained_genres = sorted(genre_counts.index.tolist())
retained_tags = sorted(tag_counts[tag_counts >= RETAINED_TAG_MIN_COUNT].index.tolist())
retained_categories = sorted(category_counts[category_counts >= RETAINED_CATEGORY_MIN_COUNT].index.tolist())

print(f"Retained genres: {len(retained_genres)}")
print(f"Retained tags with count >= {RETAINED_TAG_MIN_COUNT}: {len(retained_tags)}")
print(f"Retained categories with count >= {RETAINED_CATEGORY_MIN_COUNT}: {len(retained_categories)}")

pd.concat(
    [
        genre_counts.head(15).rename("genre_count"),
        tag_counts.head(15).rename("tag_count"),
        category_counts.head(15).rename("category_count"),
    ],
    axis=1,
)

## 5. Define modeling scopes and feature groups

The pre-release feature set is appropriate for developer pricing. The post-release set adds outcome variables for gamer-facing value assessment.

In [ ]:
ID_COLS = ["AppID", "Name"]

TARGET_COLS = [
    "current_price",
    "Discount",
    "has_active_discount",
    "estimated_list_price",
    "log_list_price",
    "price_target_source",
    "valid_price_target",
]

PRE_RELEASE_NUMERIC_FEATURES = [
    "required_age_clipped",
    "platform_count",
    "Release year",
    "release_age_years",
    "dlc_count_clipped",
    "achievements_clipped",
    "log1p_dlc_count",
    "log1p_achievements",
    "genre_count",
    "tag_count",
    "category_count",
]

PRE_RELEASE_BOOLEAN_FEATURES = ["Windows", "Mac", "Linux"]
PRE_RELEASE_MULTILABEL_FEATURES = ["genre_list", "tag_list", "category_list"]

POST_RELEASE_FEATURES = [
    "review_count",
    "positive_share",
    "log1p_review_count",
    "log1p_recommendations",
    "log1p_peak_ccu",
    "log1p_avg_playtime_forever",
    "log1p_median_playtime_forever",
    "Metacritic score",
]

RAW_REFERENCE_COLS = [
    "Release date",
    "Release date parsed",
    "Estimated owners",
    "Genres",
    "Tags",
    "Categories",
    "Developers",
    "Publishers",
]

MODEL_EXPORT_COLS = (
    ID_COLS
    + TARGET_COLS
    + RAW_REFERENCE_COLS
    + PRE_RELEASE_BOOLEAN_FEATURES
    + PRE_RELEASE_NUMERIC_FEATURES
    + PRE_RELEASE_MULTILABEL_FEATURES
    + POST_RELEASE_FEATURES
)

feature_group_summary = pd.Series({
    "id_cols": len(ID_COLS),
    "target_cols": len(TARGET_COLS),
    "pre_release_numeric": len(PRE_RELEASE_NUMERIC_FEATURES),
    "pre_release_boolean": len(PRE_RELEASE_BOOLEAN_FEATURES),
    "pre_release_multilabel": len(PRE_RELEASE_MULTILABEL_FEATURES),
    "post_release": len(POST_RELEASE_FEATURES),
    "export_cols": len(MODEL_EXPORT_COLS),
})
feature_group_summary.to_frame("count")

In [ ]:
scope_summary = pd.Series({
    "raw_rows": len(df),
    "paid_rows": df["current_price"].gt(0).sum(),
    "valid_price_target_rows": df["valid_price_target"].sum(),
    "observed_list_price_rows": df["price_target_source"].eq("observed").sum(),
    "reconstructed_list_price_rows": df["price_target_source"].eq("reconstructed").sum(),
    "free_rows": df["price_target_source"].eq("free").sum(),
    "invalid_discount_100_rows": df["price_target_source"].eq("invalid_discount_100").sum(),
    "estimated_list_price_gt_100_rows": df["estimated_list_price"].gt(100).sum(),
})
scope_summary.to_frame("count")

## 6. Save modeling table

The saved CSV keeps list columns as Python-list strings. The next notebook can parse them with `ast.literal_eval()` before multi-hot encoding.

In [ ]:
model_df = df.loc[df["valid_price_target"], MODEL_EXPORT_COLS].copy()

for col in ["genre_list", "tag_list", "category_list"]:
    model_df[col] = model_df[col].apply(repr)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(model_df):,} valid modeling rows and {model_df.shape[1]} columns to {OUTPUT_PATH}")
print("Max estimated list price:", model_df["estimated_list_price"].max())
model_df.head()

In [ ]:
reloaded = pd.read_csv(OUTPUT_PATH)
for col in ["genre_list", "tag_list", "category_list"]:
    reloaded[col] = reloaded[col].apply(ast.literal_eval)

print(f"Reloaded shape: {reloaded.shape}")
print("Valid targets after reload:", reloaded["valid_price_target"].sum())
print("Invalid targets after reload:", (~reloaded["valid_price_target"]).sum())
print("Max estimated list price after reload:", reloaded["estimated_list_price"].max())
reloaded[["AppID", "Name", "current_price", "Discount", "estimated_list_price", "genre_list", "tag_list"]].head()

## 7. Decisions for the modeling notebook

- Main target: `log_list_price`, derived from `estimated_list_price`.
- Main training scope: `valid_price_target == True`.
- Sensitivity scope: `price_target_source == "observed"` for non-discounted-only validation.
- First model: pre-release features only.
- Second model: add post-release features for gamer-facing value assessment.
- Multi-label encoding cutoffs to start with: all genres, tags with count >= 100, categories with count >= 100.